# AI Code Reviewer — end-to-end demo

This notebook runs the exact same pipeline as the CLI (`python -m ai_code_reviewer.cli sample_code/example.py --review`), but step by step so you can see each stage's output inline: static analyzers -> normalized findings -> LLM call -> structured review.

Run this notebook from the `ai-code-reviewer` project root using the project's `venv` kernel (select it in VS Code's kernel picker, top-right of the notebook).

## 1. The file being reviewed

In [ ]:
TARGET = "sample_code/iris_logistic_regression.py"

with open(TARGET, encoding="utf-8") as f:
    print(f.read())

## 2. Static analysis (pylint + flake8 + bandit), normalized

In [2]:
from ai_code_reviewer.analyzers import analyze

findings = analyze(TARGET)
findings

[{'tool': 'bandit',
  'file': '.\\sample_code/example.py',
  'line': 2,
  'column': 0,
  'code': 'B404',
  'message': 'Consider possible security implications associated with the subprocess module.',
  'severity': 'suggestion'},
 {'tool': 'bandit',
  'file': '.\\sample_code/example.py',
  'line': 11,
  'column': 0,
  'code': 'B602',
  'message': 'subprocess call with shell=True identified, security issue.',
  'severity': 'critical'},
 {'tool': 'flake8',
  'file': 'sample_code/example.py',
  'line': 1,
  'column': 1,
  'code': 'F401',
  'message': "'os' imported but unused",
  'severity': 'critical'},
 {'tool': 'pylint',
  'file': 'sample_code\\example.py',
  'line': 1,
  'column': 0,
  'code': 'C0114',
  'message': 'Missing module docstring',
  'severity': 'suggestion'},
 {'tool': 'pylint',
  'file': 'sample_code\\example.py',
  'line': 1,
  'column': 0,
  'code': 'W0611',
  'message': 'Unused import os',
  'severity': 'warning'},
 {'tool': 'pylint',
  'file': 'sample_code\\example.py'

## 3. LLM review (OpenAI gpt-4o, structured JSON output)

Requires `OPENAI_API_KEY` in `.env` at the project root (already configured).

In [3]:
from ai_code_reviewer.llm_reviewer import review_file

review = review_file(TARGET)
review

{'summary': 'The file contains several significant security issues related to subprocess usage and some code quality issues like unused imports and missing documentation. The use of shell=True in subprocess calls needs urgent attention due to command injection risks. Overall, the code could benefit from improving security practices, documentation, and adherence to naming conventions.',
 'comments': [{'line': 11,
   'severity': 'critical',
   'title': 'Security risk with subprocess usage',
   'comment': "The use of subprocess.call with shell=True, especially when coupled with user input, poses a serious command injection risk. Any unsanitized input can lead to security vulnerabilities. It's essential to explicitly pass a list of arguments to subprocess.Popen without shell=True, and properly sanitize any inputs."},
  {'line': 2,
   'severity': 'suggestion',
   'title': 'Consider security implications of subprocess',
   'comment': 'Even if the current usage here is safe, subprocess can po

## 4. Pretty-printed final review (same formatting the CLI uses)

In [4]:
severity_order = {"critical": 0, "warning": 1, "suggestion": 2}

print(f"Summary: {review['summary']}\n")
for c in sorted(review["comments"], key=lambda c: severity_order.get(c["severity"], 3)):
    print(f"[{c['severity'].upper()}] line {c['line']}: {c['title']}")
    print(f"  {c['comment']}\n")

Summary: The file contains several significant security issues related to subprocess usage and some code quality issues like unused imports and missing documentation. The use of shell=True in subprocess calls needs urgent attention due to command injection risks. Overall, the code could benefit from improving security practices, documentation, and adherence to naming conventions.

[CRITICAL] line 11: Security risk with subprocess usage
  The use of subprocess.call with shell=True, especially when coupled with user input, poses a serious command injection risk. Any unsanitized input can lead to security vulnerabilities. It's essential to explicitly pass a list of arguments to subprocess.Popen without shell=True, and properly sanitize any inputs.

[SUGGESTION] line 2: Consider security implications of subprocess
  Even if the current usage here is safe, subprocess can potentially introduce security vulnerabilities. Always consider the context in which it is used and prefer subprocess.run